In [ ]:
# BAARLE VIDEO BUILD — SINGLE CELL
# The repository is intentionally and exclusively locked to redk13/gradle.
import os, sys, json, base64, shutil, zipfile, subprocess, requests
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata, files

GH_REPO = "redk13/gradle"
GH_BRANCH = "main"
GH_TOKEN = userdata.get("GH_TOKEN")
if not GH_TOKEN:
    raise RuntimeError("Add a NEW fine-grained token to Colab Secrets as GH_TOKEN. Revoke the token previously exposed in chat.")

API = "https://api.github.com"
HEADERS = {"Authorization":f"Bearer {GH_TOKEN}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
ledger_api = f"{API}/repos/{GH_REPO}/contents/used_scripts.txt"

# FIRST PROJECT OPERATION: read the independent topic/script ledger before clone,
# pip installation, asset downloads, narration, background removal, or rendering.
r = requests.get(ledger_api + "?ref=" + GH_BRANCH, headers=HEADERS, timeout=40)
if r.status_code == 200:
    ledger_obj = r.json()
    ledger_text = base64.b64decode(ledger_obj["content"]).decode("utf-8")
    ledger_sha = ledger_obj["sha"]
elif r.status_code == 404:
    ledger_text, ledger_sha = "", None
else:
    r.raise_for_status()
    raise RuntimeError("Could not read used_scripts.txt. Stopped before all other work.")

def normalise(s):
    import re
    return " ".join(re.sub(r"[^a-z0-9]+"," ",s.lower()).split())

TOPIC = "Baarle: The Border That Runs Through Your Front Door"
expected_script = """Imagine leaving your bedroom and stepping into another country. No plane. No passport. Just slippers.

Welcome to Baarle, where Belgium and the Netherlands share one town—and the border looks like a cartographer sneezed.

In 1198, a land deal during a power struggle over Breda left plots under different authorities. The Peace of Münster gave the enclaves international status in 1648. The Belgium–Netherlands border was described in 1843.

Today, Baarle has 30 enclaves: 22 Belgian pieces surrounded by Dutch territory, plus eight Dutch pieces in the patchwork.

Seven Dutch enclaves are nested inside the two largest Belgian ones. Geography: Russian dolls, but with paperwork.

The border zigzags through streets, gardens, shops, and houses. At Loveren, it cuts right through one front door.

That house has two addresses: Belgian number 2 and Dutch number 19. Pavement markers tell you which country you're standing in.

So you can wake up in Belgium, brush your teeth in the Netherlands, and still be late for work. The twist? The enclaves weren't officially fixed until 1995—almost 800 years after the original argument. That's not border control. That's the world's slowest group project.""".strip()

if normalise(TOPIC) in normalise(ledger_text) or normalise(expected_script) in normalise(ledger_text):
    raise RuntimeError("Duplicate topic/script found in used_scripts.txt. Nothing generated. Choose an unused topic.")
print("Ledger checked first: topic and script are unused.")

# Only after the ledger check: download the exact redk13/gradle repository snapshot.
work_repo = Path("/content/gradle-work")
if work_repo.exists(): shutil.rmtree(work_repo)
work_repo.mkdir(parents=True, exist_ok=True)
z = requests.get(f"{API}/repos/{GH_REPO}/zipball/{GH_BRANCH}", headers=HEADERS, timeout=180)
z.raise_for_status()
zip_path = Path("/content/gradle-repo.zip"); zip_path.write_bytes(z.content)
with zipfile.ZipFile(zip_path) as archive: archive.extractall("/content/gradle-extract")
source = next(Path("/content/gradle-extract").iterdir())
for item in source.iterdir():
    dst = work_repo/item.name
    if item.is_dir(): shutil.copytree(item,dst)
    else: shutil.copy2(item,dst)

repo_script = (work_repo/"content"/"baarle_script.txt").read_text(encoding="utf-8").strip()
if normalise(repo_script) != normalise(expected_script):
    raise RuntimeError("Colab script does not exactly match content/baarle_script.txt in redk13/gradle.")

# GPU inference and media production are kept entirely in Colab.
subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-r",str(work_repo/"requirements-colab.txt")],check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise RuntimeError("ffmpeg/ffprobe are unavailable in this Colab runtime.")
sys.path.insert(0,str(work_repo/"src"))
from pipeline import run_pipeline

output = Path("/content/BAARLE_Border_Through_Your_Bedroom_1080x1920_60fps.mp4")
rendered = run_pipeline(str(work_repo),str(output))
if not rendered.exists() or rendered.stat().st_size < 500_000:
    raise RuntimeError("Final MP4 failed file-size QA. Ledger not updated.")

# Re-read to protect against parallel runs before appending.
latest = requests.get(ledger_api+"?ref="+GH_BRANCH,headers=HEADERS,timeout=40)
if latest.status_code == 200:
    latest_obj=latest.json()
    latest_text=base64.b64decode(latest_obj["content"]).decode("utf-8")
    latest_sha=latest_obj["sha"]
elif latest.status_code == 404:
    latest_text,latest_sha="",None
else:
    latest.raise_for_status()
    raise RuntimeError("Video rendered, but the ledger could not be re-read. Save the MP4; ledger unchanged.")

if normalise(TOPIC) in normalise(latest_text) or normalise(expected_script) in normalise(latest_text):
    raise RuntimeError("Another run recorded this topic first. Ledger left unchanged; do not overwrite.")
entry=f"TOPIC: {TOPIC}\nDATE_UTC: {datetime.now(timezone.utc).date().isoformat()}\nSCRIPT:\n{expected_script}\n{'-'*72}\n\n"
new_text=latest_text+("\n" if latest_text and not latest_text.endswith("\n") else "")+entry
body={"message":"Append completed Baarle video script to used_scripts.txt","content":base64.b64encode(new_text.encode()).decode(),"branch":GH_BRANCH}
if latest_sha: body["sha"]=latest_sha
write=requests.put(ledger_api,headers=HEADERS,json=body,timeout=60)
write.raise_for_status()
print("Render QA passed; used_scripts.txt appended in redk13/gradle.")
print(f"Final file: {rendered} ({rendered.stat().st_size:,} bytes)")
files.download(str(rendered))
